# SynBioCrow 2.6 — Seven-engine target generation (canonical RBC2 integration)

**Purpose:** run native target-specific reaction generators. This version incorporates the independently verified RetroBioCat2 notebook without replacing its native API. Other engines remain unverified/unconfigured; they are **not** claimed to work. No graph audits or closure tests.

Run cells top to bottom in Google Colab. Upload the compact ALL_OUTPUTS ZIP.


In [ ]:
from pathlib import Path
import os, subprocess, json, time
ROOT=Path('/content/rbc2_native_verified'); ROOT.mkdir(exist_ok=True)
LOG=ROOT/'logs';LOG.mkdir(exist_ok=True)
def run(args, label, timeout=1200):
    print('RUN',label,flush=True)
    with (LOG/f'{label}.log').open('w') as f:
        p=subprocess.run(args,stdout=f,stderr=subprocess.STDOUT,text=True,timeout=timeout)
    print('EXIT',p.returncode, 'LOG',LOG/f'{label}.log',flush=True)
    if p.returncode: print((LOG/f'{label}.log').read_text()[-5000:]);raise RuntimeError(label+' failed')
    return p


In [ ]:
# Locate uv robustly; do not assume its installation directory.
import shutil
from pathlib import Path
candidates = [shutil.which('uv'), '/root/.local/bin/uv', '/usr/local/bin/uv', '/usr/bin/uv', str(Path.home()/'.local/bin/uv'), str(Path.home()/'.cargo/bin/uv')]
UV = next((str(p) for p in candidates if p and Path(p).is_file() and os.access(p, os.X_OK)), None)
if UV is None:
    # Install with pip into a known isolated launcher directory; no shell PATH assumptions.
    launcher = ROOT/'uv_launcher'
    launcher.mkdir(parents=True, exist_ok=True)
    run([os.sys.executable, '-m', 'pip', 'install', '--upgrade', '--target', str(launcher), 'uv'], 'uv_install', 600)
    # The uv wheel ships a console entry point in bin/ when installed with --prefix;
    # importing uv exposes its bundled executable path reliably instead.
    import importlib.util
    uv_bin = launcher/'uv'/'uv'
    if uv_bin.is_file():
        UV = str(uv_bin)
    else:
        # Fallback: install into a known prefix and locate executable there.
        prefix=ROOT/'uv_prefix'
        run([os.sys.executable, '-m', 'pip', 'install', '--upgrade', '--prefix', str(prefix), 'uv'], 'uv_prefix_install', 600)
        found=list(prefix.rglob('bin/uv'))
        UV=str(found[0]) if found else None
if UV is None: raise RuntimeError('Could not locate uv; see uv_install.log')
print('UV executable:', UV)
run([UV, '--version'], 'uv_version', 60)
run([UV,'python','install','3.11'],'python311')
VENV=ROOT/'venv'
# Reuse an existing compatible venv; never overwrite it on repeat Colab runs.
if (VENV/'bin/python').is_file():
    probe=subprocess.run([str(VENV/'bin/python'),'-c','import sys; assert sys.version_info[:2] == (3,11)'],capture_output=True,text=True)
    if probe.returncode: raise RuntimeError('Existing RBC2 environment is incompatible; refusing to overwrite: '+probe.stderr)
    print('REUSE_EXISTING_RBC2_VENV',VENV,flush=True)
else:
    run([UV,'venv','--python','3.11',str(VENV)],'venv')
PY=str(VENV/'bin/python')
run([UV,'pip','install','--python',PY,'pip','wheel','setuptools==80.9.0'],'build_tools')
REPO=ROOT/'RetroBioCat-2'
if not REPO.exists():run(['git','clone','https://github.com/willfinnigan/RetroBioCat-2.git',str(REPO)],'clone')
run(['git','-C',str(REPO),'checkout','c5f32561b1ed99d0701a6dbb472dc448d6c082e8'],'checkout')
run([UV,'pip','install','--python',PY,'--no-deps',str(REPO)],'rbc2')
run([UV,'pip','install','--python',PY,'numpy==1.26.4','scipy==1.13.1','rdkit','scikit-learn','pandas','networkx','pyyaml','sqlalchemy','tables','h5py','openpyxl','tqdm','gdown','pytest','protobuf==4.25.9','tensorflow==2.15.1','keras==2.15.0','setuptools==80.9.0'],'dependencies',1800)
run([PY,'-c','import tensorflow, pkg_resources; from rbc2 import RetroBioCatExpander; print("RBC2_NATIVE_IMPORT_PASS")'],'smoke',300)


In [ ]:
# Change targets as needed; these are small smoke-test examples, not benchmark scores.
TARGETS=[('3-hydroxypropionic acid','O=C(O)CCO'),('butanal','CCCC=O')] # EDIT to requested target(s)
SEARCH_LITERATURE_PRECEDENT=False


In [ ]:
runner=r'''import json,traceback,sys,time,csv,hashlib,subprocess
from pathlib import Path
from rdkit import Chem
from rbc2 import RetroBioCatExpander
root=Path(sys.argv[1]);config=json.loads(Path(sys.argv[2]).read_text());root.mkdir(exist_ok=True,parents=True)
report={'status':'STARTED','engine':'RetroBioCat2','source_commit':'c5f32561b1ed99d0701a6dbb472dc448d6c082e8','search_literature_precedent':config['search_literature_precedent'],'targets':[],'timestamp':time.time()}
rows=[]
try:
 exp=RetroBioCatExpander(search_literature_precedent=config['search_literature_precedent'])
 for name,smiles in config['targets']:
  t={'name':name,'smiles':smiles,'status':'STARTED','reactions':[]};report['targets'].append(t)
  try:
   assert Chem.MolFromSmiles(smiles) is not None, 'Invalid target SMILES'
   for reaction in exp.get_reactions(smiles):
    rs=reaction.reaction_smiles();parts=rs.split('>>')
    rec={'target_name':name,'target_smiles':smiles,'reaction_smiles':rs,'reaction_name':str(getattr(reaction,'name',None)),'native_score':str(getattr(reaction,'score',None)),'engine':'RetroBioCat2','source_commit':report['source_commit'],'candidate_only':True,'reaction_smiles_parseable':len(parts)==2 and all(Chem.MolFromSmiles(x) is not None for side in parts for x in side.split('.') if x)}
    t['reactions'].append(rec);rows.append(rec)
   t['status']='NATIVE_EXECUTED';t['count']=len(t['reactions'])
  except Exception:t['status']='TARGET_FAILED';t['traceback']=traceback.format_exc()
 report['status']='COMPLETED' if all(t['status']=='NATIVE_EXECUTED' for t in report['targets']) else 'PARTIAL_FAILURE'
except Exception:report['status']='ENGINE_FAILED';report['traceback']=traceback.format_exc()
finally:
 (root/'results.json').write_text(json.dumps(report,indent=2))
 if rows:
  with (root/'synbiocrow_candidate_reactions.csv').open('w',newline='') as f:
   w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
 print(json.dumps({'status':report['status'],'counts':[(t['name'],t.get('count',0),t['status']) for t in report['targets']]},indent=2))
sys.exit(0 if report['status']=='COMPLETED' else 2)
'''
(RootRunner:=ROOT/'native_runner.py').write_text(runner)
config=ROOT/'targets.json';config.write_text(json.dumps({'targets':TARGETS,'search_literature_precedent':SEARCH_LITERATURE_PRECEDENT},indent=2))
try:run([PY,str(RootRunner),str(ROOT/'results'),str(config)],'native_expansion',900)
except Exception as e:print('Native execution failed; diagnostics will still be packaged:',e)


### Acceptance criteria
An engine counts as **verified** only when its native API produces one or more target-specific reactions for a requested compound within its supported domain, with native provenance. No fallback examples, fabricated reactions, or install-only PASS states.
